# COVID-19 Chest X-Ray Detection using Deep Learning

A reproducible Google Colab GPU workflow for a custom TensorFlow CNN. This notebook contains **no precomputed training results**. Run it only after reviewing the dataset source licenses and downloading the correct release.

**Experimental research only. Predictions are not medical diagnoses.** The three classes are COVID-19, Normal and non-COVID Pneumonia. Softmax scores are not calibrated clinical probabilities.

## 1. Start a GPU runtime and install the project

In Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Save this notebook in Drive. Clone your published project repository below; if you use a fork, edit `REPO_URL`. The repository must contain this implementation on `main`.

The Colab requirements install the GPU-capable TensorFlow package. Do not also install the CPU deployment requirements. If Colab requests a session restart after installation, restart and rerun the setup cell before proceeding.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request

REPO_URL = "https://github.com/mamta-kumari28/Chest-X-Ray-Diagnosis-using-Deep-Learning-.git"
REPO_DIR = Path("/content/Chest-X-Ray-Diagnosis-using-Deep-Learning-")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
assert Path("requirements-colab.txt").is_file(), "Publish the project files to main first."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Enable a GPU runtime before full training."
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("Free runtime storage (GiB):", round(shutil.disk_usage("/content").free / 2**30, 1))

## 2. Obtain the exact licensed dataset release

Read [`docs/DATASET.md`](../docs/DATASET.md) before downloading images. Use **COVIDx CXR-3A**, the 30,034-image three-class release associated with **Kaggle dataset version 7**. Current COVIDx releases can be binary and are incompatible with this project. Kaggle access, acceptance of applicable source terms, storage and any source-specific research permissions are external prerequisites. Source-specific license terms still apply; do not redistribute the combined images or assume unrestricted commercial use.

Download and extract the documented version into `DATA_DIR` using your own permitted Kaggle access, or mount your own Drive and point `DATA_DIR` to an extracted copy there. Never paste credentials into the notebook or commit images/patient metadata. The next cell fetches only the tiny public, pinned label files and verifies their hashes. The importer requires real upstream patient IDs; image filenames cannot replace those IDs.

The image tree may have subfolders, but each official image filename must appear exactly once. Importing fails on missing/ambiguous files, changed labels or inconsistent patient identities.

In [ ]:
# Edit this path after extracting the documented version-7 image archive.
DATA_DIR = Path("/content/datasets/covidx")
META_DIR = REPO_DIR / "data" / "metadata"
META_DIR.mkdir(parents=True, exist_ok=True)
LABEL_URL = "https://raw.githubusercontent.com/lindawangg/COVID-Net/9645505646161c7eae5fc1a8ecbfa34a77551f2f/labels/"
LABEL_HASHES = {
    "train_COVIDx_CXR-3A.txt": "a7e7ef10b00df1bdeb05c5028b04fbe99fe7011593382e2adff8a2550705835a",
    "test_COVIDx_CXR-3A.txt": "048629233e9426f1b42ce1d86e898bdb1691ff8544d81fa44b1d6a3657d88871",
}
for name, expected in LABEL_HASHES.items():
    target = META_DIR / name
    if target.exists():
        content = target.read_bytes()
    else:
        with urllib.request.urlopen(LABEL_URL + name, timeout=60) as response:
            content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected, f"Checksum mismatch: {name}"
    if not target.exists():
        target.write_bytes(content)
assert DATA_DIR.is_dir(), "Extract the licensed version-7 images, then set DATA_DIR."
print("Label files verified; image folder:", DATA_DIR)

In [ ]:
PATIENTS = META_DIR / "patients.csv"
INDEX = REPO_DIR / "data" / "processed" / "index.csv"
MANIFEST = REPO_DIR / "data" / "processed" / "manifest.csv"

# The importer preserves official patient IDs and the official train/test boundary.
# Existing patient metadata is preserved; choose a new output if switching releases.
if not PATIENTS.exists():
    subprocess.run([
        sys.executable, "scripts/import_covidx_metadata.py", "--labels",
        str(META_DIR / "train_COVIDx_CXR-3A.txt"),
        str(META_DIR / "test_COVIDx_CXR-3A.txt"),
        "--data-dir", str(DATA_DIR), "--output", str(PATIENTS),
    ], check=True)
subprocess.run([
    sys.executable, "-m", "covid_xray.data", "index",
    "--data-dir", str(DATA_DIR), "--patients", str(PATIENTS), "--output", str(INDEX),
], check=True)
subprocess.run([
    sys.executable, "-m", "covid_xray.data", "split",
    "--index", str(INDEX), "--output", str(MANIFEST),
], check=True)

## 3. Inspect the data and validate the split

The splitter keeps the official test set untouched and takes validation patients from official training data. It groups both patients and duplicate decoded images, including transitive links. If the official boundary contains a patient or exact duplicate in both pools, it stops for investigation. Near-duplicate detection, unreliable upstream IDs and cross-source identity reconciliation still need a dataset audit; hash checks cannot prove that images depict different people.

`load_manifest` verifies all image hashes and prevents cross-split patient/duplicate leakage. File decoding/hashing can take several minutes for 30,034 images. EDA is descriptive; choose architecture and hyperparameters using train/validation data only.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps

from covid_xray.config import CLASSES
from covid_xray.data import load_manifest

frame = load_manifest(MANIFEST)
counts = pd.crosstab(frame["label"], frame["split"]).reindex(CLASSES)
display(counts)
print("Images:", len(frame), "Unique upstream patients:", frame["patient_id"].nunique())
sns.countplot(data=frame, x="label", hue="split", order=list(CLASSES))
plt.title("Class distribution by patient-disjoint split")
plt.xlabel("Class")
plt.ylabel("Images")
plt.tight_layout()
plt.show()

# Use only training images for qualitative examples before model selection.
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(10, 10))
for row, label in enumerate(CLASSES):
    samples = frame[(frame["label"] == label) & (frame["split"] == "train")].sample(
        n=3, random_state=42, replace=False
    )
    for ax, path in zip(axes[row], samples["path"]):
        with Image.open(path) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train the custom CNN

The input pipeline yields resized 224 × 224 RGB pixels in `[0,255]`. **Do not divide them by 255 again**: normalization is embedded in the saved model. Small rotation, translation and zoom layers activate during training only. The CNN uses Conv2D, BatchNormalization, MaxPooling2D, Dropout and Dense layers. Class weights help with imbalance.

EarlyStopping, ModelCheckpoint and ReduceLROnPlateau monitor validation loss. The best full model, class order, real history, plots and hash-bound metadata are saved. The held-out test split is not passed to `fit`. Set `ARTIFACT_DIR` to a writable Drive folder if persistence across Colab disconnects is needed. Use a new output directory for a new experiment; `--overwrite` is available only when you intend to replace a previous model.

In [ ]:
ARTIFACT_DIR = REPO_DIR / "artifacts"
EPOCHS = 30
BATCH_SIZE = 32

subprocess.run([
    sys.executable, "-m", "covid_xray.train",
    "--manifest", str(MANIFEST), "--output", str(ARTIFACT_DIR),
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--seed", "42",
], check=True)

# These values exist only after a real successful training run.
metadata = json.loads((ARTIFACT_DIR / "model_metadata.json").read_text())
assert metadata["trained"] is True
print("Actual completed epochs:", metadata["epochs_completed"])
print("Best validation epoch:", metadata["best_epoch"])
display(Image.open(ARTIFACT_DIR / "training_curves.png"))

## 5. Evaluate the held-out test set

Run this after model selection is complete. Repeatedly tuning against these results makes the test set part of model selection; acquire a new independent test set if that happens. The evaluator requires the exact training manifest and matching model hash.

Outputs include accuracy, macro/weighted precision/recall/F1, a classification report, confusion matrix, one-vs-rest ROC-AUC when defined, per-image predictions and per-class false-positive/false-negative examples. Undefined metrics are explicitly recorded as `null` with reasons. Report the actual results and limitations; a source-confounded research benchmark is not evidence of clinical readiness.

In [ ]:
EVAL_DIR = ARTIFACT_DIR / "evaluation"
subprocess.run([
    sys.executable, "-m", "covid_xray.evaluate",
    "--manifest", str(MANIFEST), "--model", str(ARTIFACT_DIR / "model.keras"),
    "--output", str(EVAL_DIR), "--batch-size", str(BATCH_SIZE),
], check=True)
metrics = json.loads((EVAL_DIR / "metrics.json").read_text())
print(json.dumps({key: metrics[key] for key in ["accuracy", "macro", "weighted", "roc_auc_ovr"]}, indent=2))
print((EVAL_DIR / "classification_report.txt").read_text())
display(Image.open(EVAL_DIR / "confusion_matrix.png"))
for name in ["roc_curves.png", "false_predictions.png"]:
    path = EVAL_DIR / name
    if path.exists():
        display(Image.open(path))
print("Inspect error_analysis.json and individual X-rays before discussing failures.")

## 6. Export the real artifacts and run the application

Keep `model.keras`, `model_metadata.json` and `class_names.json` together. The Streamlit app verifies the model checksum and class order before loading. Upload only the model bundle to a permitted artifact host if it is too large for GitHub; use the documented model download/configuration workflow in `docs/DEPLOYMENT.md`. Dataset images, patient manifests and credentials must stay out of Git.

From an installed local checkout run `streamlit run app.py` after configuring the model path described in the deployment guide. Cloud deployment and any credentials are separate manual steps. The zip below contains generated model/evaluation artifacts only. Publicly release outputs only after checking source terms and removing patient-identifying error records; `predictions.csv` and `error_analysis.json` include dataset paths and patient IDs.

In [ ]:
# Download through Colab's Files sidebar or copy this bundle to your private Drive.
# Audit patient-level reports before sharing publicly. This does not publish anything.
bundle_path = shutil.make_archive("/content/covid-xray-artifacts", "zip", root_dir=ARTIFACT_DIR)
print("Private artifact bundle:", bundle_path)
print("No dataset images or credentials were added by this export.")